In [2]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import time
from numpy.testing import verbose
from stable_baselines3 import PPO
from sb3_contrib import TRPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy

In [3]:
### Properties of the environment

env_id = "CartPole-v1"
env_steps = 1_000_000
env_eval = 50
seed = 42
np.random.seed(seed)

In [8]:
### Training and Evaluation Loop

def train_and_eval(algo, env_id, env_steps, seed = 42, n_eval = 50):
    train_env = make_vec_env(env_id, n_envs=6, seed =seed)
    eval_env = make_vec_env(env_id,  n_envs=1, seed = int(seed*45.5))
    model = algo('MlpPolicy', train_env, verbose = 0, seed =seed)
    t0 = time.time()
    model.learn(total_timesteps = env_steps)
    mean_r, str_r = evaluate_policy(model , eval_env, n_eval_episodes = n_eval, deterministic=True)
    train_env.close()
    eval_env.close()
    return model, mean_r, str_r, t0

In [ ]:
### Training the model and saving the trained model

ppo_model, ppo_mean, ppo_str_r, ppo_t0 = train_and_eval(PPO, env_id, env_steps, seed=42, n_eval=50)
ppo_model.save('ppo_cartpole_model')
print(f'PPO Reward = {ppo_mean} +- {ppo_str_r} && time = {ppo_t0} secs')

PPO Reward = 500.0 +- 0.0 && time = 1780069816.0125556 secs


In [13]:
### Reusing the saved model

ppo_loaded = PPO.load('ppo_cartpole_model')
t0 = time.time()
env_eval = make_vec_env(env_id, n_envs=1)
mean_reward, std_dev = evaluate_policy(ppo_loaded, env_eval, n_eval_episodes=50, deterministic=True)
tf = time.time()
print(f'Reward is {mean_reward} +- {std_dev} in {tf-t0} secs approx')

Reward is 500.0 +- 0.0 in 4.105923891067505 secs approx


In [14]:
trpo_model , trpo_mean, trpo_str, trpo_t0 = train_and_eval(TRPO, env_id, env_steps, seed = 42, n_eval=50)
trpo_model.save('TRPO saved model')
print(f'The TRPO reward is {trpo_mean} +- {trpo_str} and it took {trpo_t0} secs')

The TRPO reward is 500.0 +- 0.0 and it took 1780070788.4178753 secs


In [15]:
trpo_loaded = TRPO.load('TRPO saved model')
t0 = time.time()
env_eval = make_vec_env(env_id, n_envs=1)
trpo_reward, trpo_str_r = evaluate_policy(trpo_loaded, env_eval, n_eval_episodes=50, deterministic=True)
tf = time.time()
print(f'TRPO reward = {trpo_reward} +- {trpo_str_r} && time taken during loading = {tf-t0} secs')

TRPO reward = 500.0 +- 0.0 && time taken during loading = 4.09587550163269 secs
